In [ ]:
import os
os.chdir("../")
%pwd

'c:\\Users\\Shirsh Barnwal\\OneDrive\\Desktop\\Medical Chatbot\\Medical-Chatbot'

In [ ]:
import sys
print(sys.executable)

c:\Users\Shirsh Barnwal\anaconda3\envs\medibot2\python.exe


In [ ]:
from langchain_community.document_loaders import PyPDFLoader, DirectoryLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
#Extract text from PDF files

def load_pdf_files(data):
    loader = DirectoryLoader(
    data,
    glob="**/*.pdf",
    loader_cls=PyPDFLoader
    )
    
    documents = loader.load()
    return documents

In [ ]:
%pwd

'c:\\Users\\Shirsh Barnwal\\OneDrive\\Desktop\\Medical Chatbot\\Medical-Chatbot'

In [9]:
extracted_data = load_pdf_files("data")

In [10]:
len(extracted_data)

637

In [11]:
from typing import List
from langchain_core.documents import Document

def filter_to_minimal_docs(docs: List[Document]) -> List[Document]:
    minimal_docs: List[Document] = []

    for doc in docs:
        src = doc.metadata.get("source")

        minimal_docs.append(
            Document(
                page_content=doc.page_content,
                metadata={"source": src}
            )
        )

    return minimal_docs

In [12]:
minimal_docs = filter_to_minimal_docs(extracted_data)

In [13]:
#Split Documents into chunks

def text_split(minimal_docs):
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size=1000,
        chunk_overlap=200,
        length_function=len
    )
    texts_chunks = text_splitter.split_documents(minimal_docs)
    return texts_chunks

In [14]:
text_chunks = text_split(minimal_docs)
print(f"Number of text chunks: {len(text_chunks)}")

Number of text chunks: 3428


In [15]:
from langchain_huggingface import HuggingFaceEmbeddings


def download_embeddings():
    model_name = "sentence-transformers/all-MiniLM-L6-v2"

    embeddings = HuggingFaceEmbeddings(
        model_name=model_name
    )

    return embeddings

embedding = download_embeddings()

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 1969.40it/s]


In [16]:
vector = embedding.embed_query("Hello world")
print(len(vector))


384


In [17]:
from dotenv import load_dotenv
import os
load_dotenv()

True

In [18]:
PINECONE_API_KEY = os.getenv("PINECONE_API_KEY")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY")

os.environ["PINECONE_API_KEY"] = PINECONE_API_KEY
os.environ["GOOGLE_API_KEY"] = GOOGLE_API_KEY

In [19]:
from pinecone import Pinecone
pinecone_api_key = PINECONE_API_KEY

pc = Pinecone(pinecone_api_key)

In [20]:
pc

In [21]:
from pinecone import Pinecone, ServerlessSpec

pc = Pinecone(api_key=PINECONE_API_KEY)

index_name = "medical-chatbot"

if index_name not in pc.list_indexes().names():
    
    pc.create_index(
        name=index_name,
        dimension=384,
        metric="cosine",
        spec=ServerlessSpec(
            cloud="aws",
            region="us-east-1"
        )
    )

index = pc.Index(index_name)

In [22]:
import pinecone
print(pinecone.__version__)

6.0.2


In [ ]:
'''
from langchain_pinecone import PineconeVectorStore

docsearch = PineconeVectorStore.from_documents(
    documents=text_chunks,
    embedding=embedding,
    index_name=index_name   
)
'''

In [25]:
#Load existing index

from langchain_pinecone import PineconeVectorStore
docsearch = PineconeVectorStore.from_existing_index(
    embedding=embedding,
    index_name=index_name
)

In [26]:
#add more data

dswith = Document(
page_content="dswithbappy is a youtube channel that provides tutorials on various topics.",
metadata={"source": "Youtube"}
)

In [ ]:
#docsearch.add_documents([dswith])

['21483843-6b3d-4a0b-a359-9bfb5ecc7fda']

In [28]:
retriever = docsearch.as_retriever(search_type="similarity",search_kwargs={"k": 3})

In [29]:
retrieved_docs = retriever.invoke("What is Acne?")
retrieved_docs

[Document(id='6e251285-e6a7-426e-9032-074473cd4327', metadata={'source': 'data\\02906.pdf'}, page_content='The goal of treating moderate acne is to decrease\ninflammation and prevent new comedone formation. One\neffective treatment is topical tretinoin along with a topical\nGALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin disorder in which the sebaceous\nglands become inflamed. (Photograph by Biophoto Associ-\nates, Photo Researchers, Inc. Reproduced by permission.)\nGEM - 0001 to 0432 - A  10/22/03 1:41 PM  Page 25'),
 Document(id='71962d30-b8a5-4ffb-b2f0-8a8817836d58', metadata={'source': 'data\\02906.pdf'}, page_content='The goal of treating moderate acne is to decrease\ninflammation and prevent new comedone formation. One\neffective treatment is topical tretinoin along with a topical\nGALE ENCYCLOPEDIA OF MEDICINE 2 25\nAcne\nAcne vulgaris affecting a woman’s face. Acne is the general\nname given to a skin d

In [63]:
import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

model = genai.GenerativeModel("models/gemini-2.5-flash")

In [ ]:
from langchain.chains import create_retrieval_chain
from langchain.chains.combine_documents import create_stuff_documents_chain


In [66]:
question_answer_chain = create_stuff_documents_chain(chatModel, prompt)
rag_chain = create_retrieval_chain(retriever, question_answer_chain)

In [69]:
query = "What is Acromegaly and gigantism?"

docs = retriever.invoke(query)

context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
You are a medical assistant.

Use the following context to answer the question.
If you don't know, say you don't know.

Context:
{context}

Question:
{query}
"""

response = model.generate_content(prompt)

print(response.text)

Acromegaly is a disorder where the abnormal release of growth hormone (GH) from the pituitary gland in the brain causes increased growth in bone and soft tissue, as well as other disturbances throughout the body. This specific diagnosis is given when the abnormality occurs after bone growth has stopped.

Gigantism is a variant of acromegaly that occurs in children whose bony growth plates have not yet closed. In these cases, the chemical changes of acromegaly lead to exceptional growth of long bones, resulting in unusual height.


In [72]:
query = "What is Acne? When does it occur and what are its symptoms?"

docs = retriever.invoke(query)

context = "\n\n".join([doc.page_content for doc in docs])

prompt = f"""
You are a medical assistant.

Use the following context to answer the question.
If you don't know, say you don't know.

Context:
{context}

Question:
{query}
"""

response = model.generate_content(prompt)

print(response.text)

Acne (Acne vulgaris) is the most common skin disease. It occurs when sebaceous glands, which produce oil called sebum, produce too much sebum. This excess sebum combines with dead, sticky skin cells to form a hard plug called a comedo, which blocks the skin's pores.

**When it occurs:**
Acne can arise at any age, but it usually begins at puberty and worsens during adolescence. Approximately 85% of people develop acne between the ages of 12-25 years. It can also be found in some newborns, and up to 20% of women develop mild acne. Hormonal changes can cause acne to flare up before menstruation, during pregnancy, and menopause.

**Symptoms:**
Mild noninflammatory acne consists of two types of comedones:
*   Whiteheads
*   Blackheads
The text also implies the presence of "pimples" as a symptom, as it advises against picking at them.
